In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

CREATE FLAG PARAMETER

In [0]:
dbutils.widgets.text("incremental_flag", "0")

In [0]:
incremental_flag = dbutils.widgets.get("incremental_flag")
print(incremental_flag)
print(type(incremental_flag)) 

CREATING DIMESIONS MODEL

In [0]:
%sql
select * from parquet.`abfss://silver@carsamdatalake.dfs.core.windows.net/carsales`;

Fetch Relative Columns

In [0]:

df_src = spark.sql("""
SELECT DISTINCT
  Dealer_ID AS Dealer_ID,
  DealerName
FROM parquet.`abfss://silver@carsamdatalake.dfs.core.windows.net/carsales`
""")

display(df_src)

dim_model Sink Initial and Incremental

In [0]:
%sql
DROP TABLE IF EXISTS cars_catalog.gold.dim_dealer;

In [0]:
if spark.catalog.tableExists("cars_catalog.gold.dim_dealer"):
    df_sink = spark.sql("""
SELECT
  dim_dealer_key,
  Dealer_ID,
  DealerName
FROM cars_catalog.gold.dim_dealer
""")

else:
    df_sink = spark.sql("""
SELECT
  1 AS dim_dealer_key,
  Dealer_ID,
  DealerName
FROM parquet.`abfss://silver@carsamdatalake.dfs.core.windows.net/carsales`
WHERE 1 = 0
""")

display(df_sink)

Filtering New Records and Old Records

In [0]:
df_filter = df_src.join(df_sink, df_src.Dealer_ID == df_sink.Dealer_ID, "Left")\
    .select(df_src.Dealer_ID, df_src.DealerName, df_sink.dim_dealer_key)

display(df_filter)

df_filter_old

In [0]:
df_filter_old = df_filter.filter(col("dim_dealer_key").isNotNull())

display(df_filter_old)


df_filter New

In [0]:
df_filter_new = df_filter.filter(col("dim_dealer_key").isNull()).select("Dealer_ID","DealerName")

display(df_filter_new)


CREATE SURROGATE KEY 

Fetch the max surrogate key from existing table

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("select max(dim_dealer_key) from cars_catalog.gold.dim_dealer")
    max_value = max_value_df.collect()[0][0]+1


Create surrogate key column and add max surrogate key

In [0]:
df_filter_new = df_filter_new.withColumn("dim_dealer_key", max_value+monotonically_increasing_id())
display(df_filter_new)

Create Final DF - df_filter_old + df_filter_new

In [0]:
df_final = df_filter_new.union(df_filter_old)

display(df_final)

SCD Type - 1 (Upsert)

In [0]:
from delta.tables import DeltaTable

In [0]:
%sql
DROP TABLE IF EXISTS cars_catalog.gold.dim_model;


In [0]:
# Incremental Load
if spark.catalog.tableExists("cars_catalog.gold.dim_dealer"):
    delta_tbl = DeltaTable.forPath(
        spark,
        "abfss://gold@carsamdatalake.dfs.core.windows.net/dim_dealer"
    )

    delta_tbl.alias("trg").merge(
        df_final.alias("src"),
        "trg.dim_dealer_key = src.dim_dealer_key"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

# Initial Load
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
        .option("path", "abfss://gold@carsamdatalake.dfs.core.windows.net/dim_dealer")\
        .saveAsTable("cars_catalog.gold.dim_dealer")


In [0]:
%sql
select * from cars_catalog.gold.dim_dealer;